In [ ]:
import pandas as pd

#config
cedar_file = "../../processed/cedar_combined.csv"
muller_file = "../../processed/muller_data.csv"
nepdb_file = "../../processed/NEPdb_data.csv"

In [5]:
#load datasets
cedar = pd.read_csv(cedar_file)
muller = pd.read_csv(muller_file)
nepdb = pd.read_csv(nepdb_file)

In [6]:
#standard column names
cedar.head() 

,mt_seq,wt_seq,mutation,epitope_relation,source_molecule,label
0,ACDPHSGHFV,ARDPHSGHFV,R24C,in-frame neo-epitope,NaN,1
1,AMLGTHTMEV,AMLSPHAMDV,NaN,in-frame neo-epitope,NaN,1
2,EVDPIGHLY,EVVPISHLY,NaN,in-frame neo-epitope,NaN,1
3,GVYDGEEHSV,GAYDGEEH,NaN,in-frame neo-epitope,NaN,1
4,ITKKVADLVGF,KKVADLI,NaN,in-frame neo-epitope,NaN,1


In [7]:
muller.head() 

,mt_seq,wt_seq,label,mutation,epitope_relation,source_molecule,data_source
0,DPKLKFLRL,DQKLKFLRL,0,P,SNV,KNOP1,Muller2023
1,QRNFRSVHY,QRNFRSVYY,0,H,SNV,TBC1D7,Muller2023
2,QLIPELKLL,QLIPELELL,0,K,SNV,RAD21,Muller2023
3,KPYTRIHIL,KPYTRIHIP,1,L,SNV,COPS2,Muller2023
4,YTRIHILF,YTRIHIPF,0,L,SNV,COPS2,Muller2023


In [8]:
nepdb.head()

,mt_seq,wt_seq,hla_allele,mutation,source_molecule,epitope_relation,data_source,label
0,SSGTPGRPHPGALRVAASLFLGTFF,SSGTPGRPHPGAPRVAASLFLGTFF,A*02:01,L,C1orf159,NaN,NEPdb,0
1,SSGTPGRPHPGALRVAASLFLGTFF,SSGTPGRPHPGAPRVAASLFLGTFF,A*24:01,L,C1orf159,NaN,NEPdb,0
2,VRQDNAAGRAGSNSLTQVTDLAPSL,VRQDNAAGRAGSSSLTQVTDLAPSL,A*02:01,N,MED13L,NaN,NEPdb,0
3,VRQDNAAGRAGSNSLTQVTDLAPSL,VRQDNAAGRAGSSSLTQVTDLAPSL,A*24:01,N,MED13L,NaN,NEPdb,0
4,VVGACGVGK,VVGAGGVGK,C*03:03,C,KRAS,NaN,NEPdb,0


In [11]:
#filter nepdb seqs
print("before:", len(nepdb))
nepdb = nepdb[nepdb['mt_seq'].str.len().between(8, 11)]
print("after:", len(nepdb))

before: 15912
after: 544


In [13]:
cedar_std = cedar[["mt_seq", "wt_seq", "label"]].copy()
cedar_std["data_source"] = "CEDAR"
cedar_std.head() 

,mt_seq,wt_seq,label,data_source
0,ACDPHSGHFV,ARDPHSGHFV,1,CEDAR
1,AMLGTHTMEV,AMLSPHAMDV,1,CEDAR
2,EVDPIGHLY,EVVPISHLY,1,CEDAR
3,GVYDGEEHSV,GAYDGEEH,1,CEDAR
4,ITKKVADLVGF,KKVADLI,1,CEDAR


In [14]:
prime_std = muller[["mt_seq", "wt_seq", "label"]].copy()
prime_std["data_source"] = "PRIME"
prime_std.head()

,mt_seq,wt_seq,label,data_source
0,DPKLKFLRL,DQKLKFLRL,0,PRIME
1,QRNFRSVHY,QRNFRSVYY,0,PRIME
2,QLIPELKLL,QLIPELELL,0,PRIME
3,KPYTRIHIL,KPYTRIHIP,1,PRIME
4,YTRIHILF,YTRIHIPF,0,PRIME


In [15]:
nepdb_stad = nepdb[["mt_seq", "wt_seq", "label"]].copy()
nepdb_stad["data_source"] = "NEPdb"
nepdb_stad.head()

,mt_seq,wt_seq,label,data_source
4,VVGACGVGK,VVGAGGVGK,0,NEPdb
5,VVGACGVGK,VVGAGGVGK,0,NEPdb
6,VVGACGVGK,VVGAGGVGK,0,NEPdb
7,VVGACGVGK,VVGAGGVGK,0,NEPdb
8,VVGACGVGK,VVGAGGVGK,0,NEPdb


In [16]:
#KEEP ONLY POSITIVE FROM EXTERNAL
prime_pos = prime_std[prime_std['label'] == 1]
nepdb_pos = nepdb_stad[nepdb_stad['label'] == 1]

print("prime pos:", len(prime_pos))
print("nepdb pos:", len(nepdb_pos))


prime pos: 178
nepdb pos: 276


In [17]:
#combine with CEDAR
combined = pd.concat([cedar_std, prime_pos, nepdb_pos], ignore_index=True)
combined.head() 

,mt_seq,wt_seq,label,data_source
0,ACDPHSGHFV,ARDPHSGHFV,1,CEDAR
1,AMLGTHTMEV,AMLSPHAMDV,1,CEDAR
2,EVDPIGHLY,EVVPISHLY,1,CEDAR
3,GVYDGEEHSV,GAYDGEEH,1,CEDAR
4,ITKKVADLVGF,KKVADLI,1,CEDAR


In [18]:
#remove duplicates
combined = combined.drop_duplicates(subset=["mt_seq"])

#reset index
combined = combined.reset_index(drop=True)
combined.head() 

,mt_seq,wt_seq,label,data_source
0,ACDPHSGHFV,ARDPHSGHFV,1,CEDAR
1,AMLGTHTMEV,AMLSPHAMDV,1,CEDAR
2,EVDPIGHLY,EVVPISHLY,1,CEDAR
3,GVYDGEEHSV,GAYDGEEH,1,CEDAR
4,ITKKVADLVGF,KKVADLI,1,CEDAR


In [19]:
#save
combined.to_csv("../processed/combined_positives.csv", index=False)

In [20]:
#summary
print("Final combined size:", len(combined))
print(combined["label"].value_counts())

Final combined size: 11760
label
0    9103
1    2657
Name: count, dtype: int64
